In [36]:
import pandas as pd
import numpy as np
import os
import logging
from typing import List, Dict

In [37]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("data_pipeline.log", mode="w", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)

columns_to_keep:List[str]=[
    'region', 'parent_category_name', 'category_name',
    'param_1', 'param_2', 'param_3', 'price', 'item_seq_number',
    'user_type', 'description', 'image', 'deal_probability'
]

dtypes:Dict[str, str]={
    'region': 'category',
    'parent_category_name': 'category',
    'category_name': 'category',
    'user_type': 'category',
    'price': 'float32',
    'item_seq_number': 'int32',
    'deal_probability': 'float32'
}

In [38]:
def load_raw_data(file_path:str, nrows:int=5000)->pd.DataFrame:
    if not os.path.exists(file_path):
        logger.error(f"File {file_path} is not found")
        raise FileNotFoundError(f"Missing file {file_path}")
    

    try:
        df=pd.read_csv(file_path, usecols=columns_to_keep,dtype=dtypes, nrows=nrows)
        logger.info(f"File {file_path} is successfully found and loaded")
        return df
    
    except Exception as e:
        logger.info(f"Error occured during data ingestion:{e} ")
        raise

In [39]:
def feature_engineering(df:pd.DataFrame)->pd.DataFrame:
    processed_df=df.copy()

    processed_df['has_params']=processed_df[['param_1', 'param_2', 'param_3']].notna().any(axis=1).astype('int8')

    processed_df['param_1_exists']=processed_df['param_1'].notna().astype('int8')
    processed_df['param_2_exists']=processed_df['param_2'].notna().astype('int8')
    processed_df['param_3_exists']=processed_df['param_3'].notna().astype('int8')

    processed_df['param_1']=processed_df['param_1'].fillna("")
    processed_df['param_2']=processed_df['param_2'].fillna("")
    processed_df['param_3']=processed_df['param_3'].fillna("")

    processed_df['image']=processed_df['image'].notna().astype('int8')

    processed_df['description']=processed_df['description'].fillna("")
    processed_df['description_len']=processed_df['description'].str.len().astype('int32')

    bins=[-1, 10, 50, 250, 1000, np.inf]
    labels=['1. Пусто', '2. Очень короткое', '3. Оптимальное', '4. Подробное', '5. Слишком длинное']

    processed_df['description_group']=pd.cut(processed_df['description_len'], bins=bins, labels=labels)

    logger.info("Feature Engineering completed successfully")

    return processed_df

In [40]:
def removing_outliers(df:pd.DataFrame, 
                      target_value:str='price', 
                      group_col:str='category_name'
                      )->pd.DataFrame:
    q_low = df.groupby(group_col)[target_value].transform('quantile', 0.01)
    q_high = df.groupby(group_col)[target_value].transform('quantile', 0.99)

    df_cleaned= df[
        ((df['price'] >= q_low) & (df['price'] <= q_high)) | 
        df['price'].isnull()
    ]

    dropped_count=len(df)-len(df_cleaned)

    logger.info(f"Outlier removal complete. Dropped {dropped_count} rows out of bounds")
    return df_cleaned

In [41]:
def run_pipeline(input_file:str, output_file:str)->None:
    try:
        raw_df=load_raw_data(input_file)
        engineered_df=feature_engineering(raw_df)
        final_df=removing_outliers(engineered_df)

        if 'description' in final_df.columns:
            final_df=final_df.drop(columns=['description'])
        
        final_df.to_csv(output_file, index=False)

    except Exception as e:
        logger.critical(f"Pipeline crashed during execution. Review logs. Details: {e}")

In [42]:
if __name__ == "__main__":
    INPUT_PATH = "../train.csv"
    OUTPUT_PATH = "train_prepared1.csv"
    
    run_pipeline(INPUT_PATH, OUTPUT_PATH)

2026-06-11 22:02:03,931 - INFO - File ../train.csv is successfully found and loaded
2026-06-11 22:02:03,934 - INFO - Feature Engineering completed successfully
2026-06-11 22:02:03,940 - INFO - Outlier removal complete. Dropped 128 rows out of bounds
